# Load Data

In [1]:
import pandas as pd

In [2]:
data=pd.read_csv("IMDB Dataset.csv")

In [3]:
data.shape

(50000, 2)

In [4]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
data.isnull().sum()

,0
review,0
sentiment,0


In [6]:
data.drop_duplicates(inplace=True)

In [7]:
data.shape

(49582, 2)

# Preprocessing

In [8]:
# 1) Converting to lower case
data["review"]=data["review"].str.lower()

In [9]:
# 2)Removing URLs
import re
def remove_urls(text):
    text= re.sub(r"https/S+","",text)
    return text
data["review"]=data["review"].apply(remove_urls)

In [10]:
# 3)Removing Puntuations
def remove_puntuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
data["review"]=data["review"].apply(remove_puntuations)

In [11]:
data.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [12]:
# 4)Removing HTML
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
data["review"]=data["review"].apply(remove_html)

In [13]:
# 5) Stopwords
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")

  return text

data["review"]=data["review"].apply(remove_stopwords)

In [16]:
data.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [17]:
# 6)stemming
from nltk.stem import PorterStemmer
def Stemming(text):
  ps=PorterStemmer()
  stemmed_words=[]
  tokens=word_tokenize(text)
  for token in tokens:
    stemmed_words.append(ps.stem(token))

  return " ".join(stemmed_words)

In [18]:
data["review"]=data["review"].apply(Stemming)

In [19]:
data.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [20]:
# Encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
data["sentiment"]=le.fit_transform(data["sentiment"])

In [21]:
y=data["sentiment"]

In [22]:
y.head()

,sentiment
0,1
1,1
2,1
3,0
4,1


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(data["review"])

In [25]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057158 stored elements and shape (49582, 5000)>

# Build RNN

In [26]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [27]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [28]:
X_train.shape

(39665, 5000)

In [29]:
X_test.shape

(9917, 5000)

In [31]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [32]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(Y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(Y_test.values).float()
)

In [33]:
trainLoader=DataLoader(train_set,shuffle=True,batch_size=64)
testLoader=DataLoader(test_set,shuffle=True,batch_size=64)

In [34]:
import torch.nn as nn
import torch.optim as optim

In [40]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):
    super().__init__()

    self.hidden_size=hidden_size
    self.num_layers=num_layers

    self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
    self.fc=nn.Linear(hidden_size,1)

  def forward(self,x):
    h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
    out,_=self.rnn(x,h0)
    output=self.fc(out[:,-1,:])
    return output

In [42]:
input_size=X_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [43]:
epochs=10
for epoch in range(epochs):
  model.train()
  for xb,yb in trainLoader:
    optimizer.zero_grad()
    xb=xb.unsqueeze(1)
    outputs=model(xb)
    outputs=torch.sigmoid(outputs.squeeze())

    loss=criterion(outputs,yb)
    loss.backward()
    optimizer.step()
  print(f"Epoch: {epoch}/{epochs} and loss= {loss.item()}")



Epoch: 0/10 and loss= 0.17984744906425476
Epoch: 1/10 and loss= 0.2545391023159027
Epoch: 2/10 and loss= 0.36535996198654175
Epoch: 3/10 and loss= 0.24552549421787262
Epoch: 4/10 and loss= 0.1905360370874405
Epoch: 5/10 and loss= 0.41993486881256104
Epoch: 6/10 and loss= 0.18329094350337982
Epoch: 7/10 and loss= 0.19358308613300323
Epoch: 8/10 and loss= 0.3456208407878876
Epoch: 9/10 and loss= 0.2957392632961273


In [46]:
model.eval()
with torch.no_grad():
  correct=0
  total=0
  for xb,yb in testLoader:
    xb=xb.unsqueeze(1)
    outputs=model(xb)
    predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()
    total+=yb.size(0)
    correct+=(predicted==yb).sum().item()
  print(f"Accuracy: {correct/total*100}")

Accuracy: 85.68115357466975
